<a href="https://colab.research.google.com/github/LBDillon/decoding-design-bias/blob/main/score_proteins_esm3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESM3-open (1.4B) scoring for decoding-design-bias

Scores every protein in `Decoding_Bias_Dataset.csv` under two modes, both using iterative masked pseudo-log-likelihood (mask one residue at a time, read native log-prob):

| Mode | Structure track | Comparable to |
| --- | --- | --- |
| `esm3_struct_cond_score` | VQ-VAE tokens from AF backbone, kept intact | PiFold, ProteinMPNN, ESM-IF |
| `esm3_seq_only_score`    | masked (no structure info)                 | ESM2-15B pppl, CARP |

The within-ESM3 struct vs no-struct contrast is the clean addition: same architecture, same tokenizer, same scoring, structure conditioning flipped on/off.

**Runtime on A100 (bf16):** ~8–12h for 7843 proteins × 2 modes. Runs with a resume-by-Entry checkpoint so disconnects are cheap.

## 1. Setup: GPU, deps, HF auth

In [ ]:
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-ef8ceb14-067e-9c20-52c3-6bb087f5fabe)


In [ ]:
!pip install -q esm biopython requests tqdm

In [ ]:
# Accept the gated repo at https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1 first.
from huggingface_hub import login
login()  # paste token inline

## 2. Mount Drive, clone repo, config paths

Output CSV is written to Drive so progress survives runtime disconnects. Dataset comes from the public GitHub repo (7.5MB). PDBs are fetched on-the-fly from the AlphaFold DB into ephemeral `/content/pdbs/` (they don't need to persist).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/cohort_pdb_scoring_inputs.csv'

# Drive output location — survives runtime disconnects
# Updated to reflect PDB-based structures
DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding_bias_results/PDB/esm3'
OUTPUT        = f'{DRIVE_OUT_DIR}/esm3_pdb_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

PDB_CACHE = '/content/pdbs'
os.makedirs(PDB_CACHE, exist_ok=True)
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)

In [ ]:
!unzip /content/cohort_pdb_scoring_bundle.zip -d /content/

## 3. Load model (≈ 1 min; 5GB download first time)

In [ ]:
import torch
from esm.models.esm3 import ESM3

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

model = ESM3.from_pretrained('esm3-open').to(device).eval()
tokenizers = model.tokenizers
mask_id = tokenizers.sequence.mask_token_id
print('mask_id =', mask_id, ' | param count:', sum(p.numel() for p in model.parameters())/1e9, 'B')

## 4. PDB fetching + scoring functions

In [ ]:
import zipfile
import os
"""


# Path to the zip file provided by the user
ZIP_FILE_PATH = '/content/cohort_pdb_scoring_bundle.zip' # Corrected zip file name

# Ensure the PDB_CACHE directory exists (PDB_CACHE is defined in a previous cell)
os.makedirs(PDB_CACHE, exist_ok=True)

# Define the *actual* directory where PDB files will reside after extraction.
# The zip file contains a folder called 'cohort_chain_structs' at its root.
# When extracted to PDB_CACHE ('/content/pdbs'), the PDBs will be in '/content/pdbs/cohort_chain_structs'.
PDB_FILES_LOCATION = os.path.join(PDB_CACHE, 'cohort_chain_structs')
os.makedirs(PDB_FILES_LOCATION, exist_ok=True) # Ensure this target directory exists

# Extract the zip file contents if the PDB_FILES_LOCATION is empty
if not os.listdir(PDB_FILES_LOCATION): # Check if the *target* directory is empty
    print(f"Extracting {ZIP_FILE_PATH} to {PDB_CACHE} (contents will be in {PDB_FILES_LOCATION})...")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(PDB_CACHE) # This will create 'cohort_chain_structs' inside PDB_CACHE
    print("Extraction complete.")
else:
    print(f"PDB files already exist in {PDB_FILES_LOCATION}. Skipping extraction.")
"""
PDB_FILES_LOCATION = "/content/cohort_chain_structs"
def fetch_pdb(row):
    """Fetches a PDB file by looking for it in the local extracted directory,
    using the full chain_pdb_path from the dataset row."""
    if 'chain_pdb_path' not in row:
        return None

    # Extract the base filename from the chain_pdb_path in the dataset
    expected_filename = os.path.basename(row['chain_pdb_path'])

    # Construct the full local path using the determined PDB_FILES_LOCATION
    local_path = os.path.join(PDB_FILES_LOCATION, expected_filename)

    if os.path.exists(local_path):
        return local_path
    # If not found, return None
    return None

In [ ]:
import torch.nn.functional as F
from esm.sdk.api import ESMProtein
from esm.utils.structure.protein_chain import ProteinChain
@torch.no_grad()
def score_protein(pdb_path, max_tokens_per_batch=2048):
    chain = ProteinChain.from_pdb(pdb_path)
    if not chain.sequence:
        return None

    protein = ESMProtein.from_protein_chain(chain)
    pt = model.encode(protein)

    seq_tokens = pt.sequence.to(device)
    struct_tokens = pt.structure.to(device) if pt.structure is not None else None
    L = seq_tokens.shape[0] - 2
    native = seq_tokens[1:1 + L]

    def _run(batch_size):
        results = {}
        for mode_name, use_struct in [('struct_cond', True), ('seq_only', False)]:
            lps = torch.zeros(L, device=device)
            for start in range(0, L, batch_size):
                end = min(start + batch_size, L)
                B = end - start
                seq_batch = seq_tokens.unsqueeze(0).repeat(B, 1).clone()
                for j in range(B):
                    seq_batch[j, start + j + 1] = mask_id
                struct_batch = (struct_tokens.unsqueeze(0).repeat(B, 1)
                                if use_struct and struct_tokens is not None else None)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    esmout = model.forward(sequence_tokens=seq_batch,
                                           structure_tokens=struct_batch)
                logits = esmout.sequence_logits.float()
                for j in range(B):
                    pos = start + j + 1
                    lp = F.log_softmax(logits[j, pos], dim=-1)
                    lps[start + j] = lp[native[start + j]]
                del esmout, logits
            results[f'{mode_name}_mean'] = lps.mean().item()
            results[f'{mode_name}_sum']  = lps.sum().item()
        return results

    batch_size = max(1, max_tokens_per_batch // (L + 2))
    while True:
        try:
            out = _run(batch_size)
            out['length'] = L
            return out
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f'  OOM, retrying L={L} batch={batch_size}')

    out['length'] = L
    return out

## 5. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    t0 = time.time()
    # Pass the full row 'r' to fetch_pdb as it contains 'chain_pdb_path'
    pdb = fetch_pdb(r)
    t_dl = time.time() - t0
    if pdb is None:
        print(entry, 'no PDB'); continue
    t0 = time.time()
    res = score_protein(pdb)
    t_sc = time.time() - t0
    print(f'{entry} L={res["length"]} struct_cond={res["struct_cond_mean"]:.4f} seq_only={res["seq_only_mean"]:.4f}  dl={t_dl:.1f}s score={t_sc:.1f}s')

In [ ]:
print('Listing a few files from the PDB_CACHE directory to check naming conventions:')
# List up to 10 files from the cache directory
import os
for i, filename in enumerate(os.listdir(PDB_CACHE)):
    if i >= 10:
        break
    print(filename)

# Also, let's inspect the 'chain_pdb_path' from the dataset for one of the missing entries
# For example, for '5ZX1_1' from the 'rows' variable in the kernel state
missing_entry_example = next((r for r in rows if r['Entry'] == '5ZX1_1'), None)
if missing_entry_example:
    print(f"\nExpected path from dataset for '5ZX1_1': {missing_entry_example['chain_pdb_path']}")
    # Extract just the filename to compare with the cached files
    expected_filename = os.path.basename(missing_entry_example['chain_pdb_path'])
    print(f"Expected filename from dataset for '5ZX1_1': {expected_filename}")

    # Check if this expected filename exists in the PDB_CACHE
    if os.path.exists(os.path.join(PDB_CACHE, expected_filename)):
        print(f"Found '{expected_filename}' in cache using its full path.")
    else:
        print(f"'{expected_filename}' not found directly in cache using its full path. This confirms a naming mismatch.")

## 6. Full run with resume

Safe to re-run — it picks up where it left off based on entries already in `OUTPUT`. Flushes after every protein so a disconnect only loses the in-flight one.

In [ ]:
from tqdm.auto import tqdm

from tqdm.auto import tqdm

already = set()
# By setting open_mode to 'w', we force an overwrite regardless of existing data.
# The 'if os.path.exists(OUTPUT):' block will still read existing data for 'already' set
# if we want to retain that logic. However, for a full overwrite, we explicitly set mode to 'w'.
# Let's remove the logic that reads existing data if we intend a full overwrite from scratch.
# If you just want to restart and not resume, the 'already' set should be empty.

# To force a complete overwrite, we can skip loading 'already' and explicitly set open_mode.
# For this case, we want to clear the previous results.

# If you truly want to overwrite, it's simpler to set `open_mode = 'w'` directly.
# This will truncate the file if it exists, and the `already` set will remain empty,
# thus processing all `todo` items.

# We will also remove the `already` loading logic to ensure a fresh start.
# Let's assume a full overwrite is desired, so `already` should be an empty set.
already = set()

todo = [r for r in rows if r['Entry'] not in already]
todo = [
    r for r in todo
    if int(float(r.get('sequence_length') or len(r.get('sequence', '')))) <= 1500
]

print('filtered to', len(todo), 'proteins ≤1500aa')
print('remaining:', len(todo))

# Force open_mode to 'w' to overwrite the file
open_mode = 'w'
with open(OUTPUT, open_mode, newline='') as out:
    w = csv.writer(out)
    if open_mode == 'w': # This condition will now always be true
        w.writerow([
            'Entry', 'species', 'domain',
            'esm3_struct_cond_score', 'esm3_seq_only_score',
            'esm3_struct_cond_sum', 'esm3_seq_only_sum',
            'scored_length', 'dataset_length',
        ])

    counts = {'ok': 0, 'missing_pdb': 0, 'error': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='esm3'):
        entry = r['Entry']
        # Pass the full row 'r' to fetch_pdb as it contains 'chain_pdb_path'
        pdb = fetch_pdb(r)
        if pdb is None:
            counts['missing_pdb'] += 1
            continue
        try:
            res = score_protein(pdb)
        except Exception as exc:
            print(f'[{entry}] {exc}')
            counts['error'] += 1
            continue
        if res is None:
            counts['error'] += 1
            continue
        w.writerow([
            entry, r.get('species', ''), r.get('domain', ''),
            f"{res['struct_cond_mean']:.6f}", f"{res['seq_only_mean']:.6f}",
            f"{res['struct_cond_sum']:.6f}", f"{res['seq_only_sum']:.6f}",
            res['length'], len(r.get('sequence', '')),
        ])
        out.flush()
        torch.cuda.empty_cache()
        counts['ok'] += 1

print('done in', round(time.time() - t_start, 1), 's', counts)

filtered to 876 proteins ≤1500aa
remaining: 876


esm3:   0%|          | 0/876 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/esm/models/vqvae.py:283: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: ignore
/usr/local/lib/python3.12/dist-packages/esm/models/vqvae.py:172: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return data[ranges]
/usr/local/lib/python3.12/dist-packages/esm/models/vqvae.py:283: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: 

OSError: [Errno 107] Transport endpoint is not connected

## 7. Quick look at results

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
df[['esm3_struct_cond_score','esm3_seq_only_score']].describe()